In [4]:
## baseline BMI and calorie calculation formula

def calculate_bmi(weight, height):
    height_m = height / 100  # convert cm → m
    return weight / (height_m ** 2)

def calculate_bmr(weight, height, age, gender):
    if gender.lower() == "male":
        return 10 * weight + 6.25 * height - 5 * age + 5
    else:
        return 10 * weight + 6.25 * height - 5 * age - 161


# Example (Male, 25 years, 70kg, 175cm)
bmi = calculate_bmi(70, 175)
bmr = calculate_bmr(70, 175, 25, "Male")


print("BMI:", round(bmi, 2))
print("BMR:", round(bmr, 2))




BMI: 22.86
BMR: 1673.75


In [2]:
## data cleaning and preprocesssing

import pandas as pd

# Load the dataset user uploaded
file_path = 'diet_users_10000.csv'
df = pd.read_csv(file_path)

# Preview the first few rows and datatypes
df.info(), df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   name            10000 non-null  object 
 1   age             10000 non-null  int64  
 2   weight          10000 non-null  float64
 3   height          10000 non-null  float64
 4   target(weight)  10000 non-null  float64
 5   language        10000 non-null  object 
 6   gender          10000 non-null  object 
 7   time            10000 non-null  object 
 8   diet_type       10000 non-null  object 
dtypes: float64(3), int64(1), object(5)
memory usage: 703.3+ KB


(None,
          name  age  weight  height  target(weight) language gender       time  \
 0  Aditya9336   34    98.2   156.6            81.2  Chinese   Male  16 months   
 1    Diya1413   25    59.0   197.8            60.9   German  Other  15 months   
 2    Diya8775   49    44.7   187.7            69.8  Spanish  Other   2 months   
 3    Riya1866   63   119.2   194.9            89.2   German   Male   1 months   
 4  Ananya9027   60    75.5   193.3            69.0    Hindi  Other   8 months   
 
     diet_type  
 0  Eggitarian  
 1         Veg  
 2  Eggitarian  
 3     Non-Veg  
 4     Non-Veg  )

In [ ]:
import numpy as np

# time 
df['time_months'] = df['time'].str.extract(r'(\d+)').astype(int)

# encode the categorial features
df['gender_encoded'] = df['gender'].map({'Male': 0, 'Female': 1, 'Other': 2})
df['diet_encoded'] = df['diet_type'].map({'Veg': 0, 'Non-Veg': 1, 'Eggitarian': 2})

# Bmi
df['BMI'] = df['weight'] / ((df['height'] / 100) ** 2)

# < 18.5 → Underweight

# 18.5 – 24.9 → Normal

# 25 – 29.9 → Overweight

# ≥ 30 → Obese

# BMI → helps classify users (underweight/overweight).

# BMR → used to calculate daily calorie intake.

## BMr
def bmr(row):
    if row['gender'] == 'Male':
        return 10*row['weight'] + 6.25*row['height'] - 5*row['age'] + 5
    elif row['gender'] == 'Female':
        return 10*row['weight'] + 6.25*row['height'] - 5*row['age'] - 161
    else:  # default neutral
        return 10*row['weight'] + 6.25*row['height'] - 5*row['age']

df['BMR'] = df.apply(bmr, axis=1)


## calorie to reach  target  

df['calorie_adjustment'] = ((df['weight'] - df['target(weight)']) * 7700) / (df['time_months'] * 30)

## if calorie adjustment is negative, then the user needs to increase its calorie intake
## if calorie adjustment is positive, then the user needs to decrease its calorie intake





df[['name','age','weight','height','BMI','BMR','calorie_adjustment','gender_encoded','diet_encoded']].head()

,name,age,weight,height,BMI,BMR,calorie_adjustment,gender_encoded,diet_encoded
0,Aditya9336,34,98.2,156.6,40.043126,1795.750,272.708333,0,2
1,Diya1413,25,59.0,197.8,15.079934,1701.250,-32.511111,2,0
2,Diya8775,49,44.7,187.7,12.687585,1375.125,-3221.166667,2,2
3,Riya1866,63,119.2,194.9,31.379974,2100.125,7700.000000,0,1
4,Ananya9027,60,75.5,193.3,20.206135,1663.125,208.541667,2,1
